# Notebook 3 - Preliminary Dataset Visualization

This notebook provides an exploratory view of the historical 5-minute XAU/USD dataset used in the study. It generates the preliminary visualizations presented in Chapter III before the finalized LGMMA-X modeling and evaluation workflow.

The notebook is intended for exploratory analysis only and is **not part of the finalized LGMMA-X system pipeline**. The visualizations produced here are descriptive and should not be interpreted as anomaly-detection outputs or final model results.

## Figures generated

1. Historical 5-Minute XAU/USD Closing Price Series
2. Five-Minute XAU/USD Return Behavior
3. Rolling Volatility of Five-Minute XAU/USD Returns
4. Distribution of Five-Minute XAU/USD Returns


## 1. Imports and configuration

The notebook uses Pandas and NumPy for data handling and Matplotlib for visualization. The configuration cell allows the raw CSV path to be changed without modifying the analysis code.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Run this notebook from the repository's notebooks/ directory.
REPO_ROOT = Path("..")
RAW_DATA_DIR = REPO_ROOT / "pipeline" / "data" / "raw"

# Set this explicitly when more than one CSV exists in RAW_DATA_DIR.
DATA_PATH = None

ROLLING_WINDOW = 12   # 12 x 5-minute candles = 1 hour
GAP_MINUTES = 5

print(f"Raw-data directory: {RAW_DATA_DIR.resolve()}")


Raw-data directory: C:\Users\Kyle Eva\Downloads\datasci2-thesis\src\notebooks\pipeline\data\raw


## 2. Locate and load the ForexSB export

The ForexSB MetaTrader-style export does not contain column headers. The expected columns are therefore assigned explicitly as `Date`, `Open`, `High`, `Low`, `Close`, and `Timeframe`.

The final `Timeframe` column is retained only for source-file compatibility and is not used in the visualizations.


In [4]:
EXPECTED_COLUMNS = [
    "Date",
    "Open",
    "High",
    "Low",
    "Close",
    "Timeframe",
]


def resolve_data_path() -> Path:
    """Resolve the raw CSV path used by the notebook."""
    if DATA_PATH is not None:
        path = Path(DATA_PATH)
        if not path.exists():
            raise FileNotFoundError(
                f"Configured DATA_PATH does not exist: {path.resolve()}"
            )
        return path

    candidates = sorted(RAW_DATA_DIR.glob("*.csv"))

    if not candidates:
        raise FileNotFoundError(
            f"No CSV files were found in {RAW_DATA_DIR.resolve()}. "
            "Set DATA_PATH in the configuration cell."
        )

    if len(candidates) > 1:
        names = "\n".join(f"- {path.name}" for path in candidates)
        raise RuntimeError(
            "Multiple CSV files were found. Set DATA_PATH explicitly:\n"
            + names
        )

    return candidates[0]


data_path = resolve_data_path()
print(f"Loading: {data_path.resolve()}")

df = pd.read_csv(
    data_path,
    header=None,
    names=EXPECTED_COLUMNS,
)

df["Date"] = pd.to_datetime(
    df["Date"],
    utc=True,
    errors="coerce",
)

for column in ["Open", "High", "Low", "Close"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = (
    df.dropna(subset=["Date", "Open", "High", "Low", "Close"])
      .sort_values("Date")
      .drop_duplicates(subset=["Date"])
      .reset_index(drop=True)
)

print(f"Rows loaded: {len(df):,}")
print(f"Earliest timestamp: {df['Date'].min()}")
print(f"Latest timestamp:   {df['Date'].max()}")

display(df.head())


FileNotFoundError: No CSV files were found in C:\Users\Kyle Eva\Downloads\datasci2-thesis\src\notebooks\pipeline\data\raw. Set DATA_PATH in the configuration cell.

## 3. Initial dataset summary

The summary below provides the basic coverage and structure of the imported XAU/USD observations before the exploratory calculations.


In [ ]:
print("Dataset summary")
print("=" * 60)
print(f"Records:        {len(df):,}")
print(f"Start (UTC):    {df['Date'].min()}")
print(f"End (UTC):      {df['Date'].max()}")
print(f"Unique dates:   {df['Date'].dt.date.nunique():,}")

print("\nPrice summary")
display(df[["Open", "High", "Low", "Close"]].describe().T)


## 4. Identify temporal gaps for exploratory calculations

The raw observations are inspected for intervals larger than five minutes. For the exploratory return and rolling-volatility calculations, continuous timestamp segments are treated independently so that an overnight, holiday, or provider-related gap does not create an artificial return spanning the missing interval.


In [ ]:
time_diff = df["Date"].diff()
gap = time_diff > pd.Timedelta(minutes=GAP_MINUTES)

df["segment_id"] = gap.cumsum()
df["gap_minutes"] = time_diff.dt.total_seconds().div(60)

gap_rows = df.loc[gap, ["Date", "gap_minutes", "segment_id"]].copy()

print(f"Detected temporal gaps: {len(gap_rows):,}")
print(f"Continuous segments:    {df['segment_id'].nunique():,}")

if not gap_rows.empty:
    display(gap_rows.head(10))


## 5. Compute exploratory returns and rolling volatility

The five-minute log return is calculated independently within each continuous segment. Rolling volatility is computed using the same 12-candle window used by the finalized feature-engineering logic, corresponding to approximately one hour of 5-minute observations.


In [ ]:
df["log_return"] = (
    df.groupby("segment_id")["Close"]
      .transform(lambda series: np.log(series / series.shift(1)))
)

df["rolling_volatility"] = (
    df.groupby("segment_id")["log_return"]
      .transform(lambda series: series.rolling(window=ROLLING_WINDOW).std())
)

print("Exploratory features calculated.")
display(
    df[["Date", "Close", "log_return", "rolling_volatility"]].head(20)
)


## 6. Figure III-1 — Historical 5-Minute XAU/USD Closing Price Series

The closing-price series provides an overall view of the historical movement of XAU/USD across the collected period. This visualization is descriptive and is not used directly as an anomaly score.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df["Date"], df["Close"], linewidth=0.8)
ax.set_title("Historical 5-Minute XAU/USD Closing Price Series")
ax.set_xlabel("Timestamp (UTC)")
ax.set_ylabel("Closing Price (USD)")
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()


**Figure III-1. Historical 5-Minute XAU/USD Closing Price Series**


## 7. Figure III-2 — Five-Minute XAU/USD Return Behavior

The five-minute log-return series highlights short-term price changes between consecutive valid observations within each continuous segment.


In [ ]:
return_plot = df.dropna(subset=["log_return"])

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(
    return_plot["Date"],
    return_plot["log_return"],
    linewidth=0.6,
)
ax.axhline(0, linewidth=0.8)
ax.set_title("Five-Minute XAU/USD Return Behavior")
ax.set_xlabel("Timestamp (UTC)")
ax.set_ylabel("Log Return")
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()


**Figure III-2. Five-Minute XAU/USD Return Behavior**


## 8. Figure III-3 — Rolling Volatility of Five-Minute XAU/USD Returns

Rolling volatility is calculated as the rolling standard deviation of five-minute log returns using a 12-candle window. This provides an exploratory view of how short-term variability changes across the historical period.


In [ ]:
volatility_plot = df.dropna(subset=["rolling_volatility"])

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(
    volatility_plot["Date"],
    volatility_plot["rolling_volatility"],
    linewidth=0.8,
)
ax.set_title("Rolling Volatility of Five-Minute XAU/USD Returns")
ax.set_xlabel("Timestamp (UTC)")
ax.set_ylabel("Rolling Standard Deviation")
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()


**Figure III-3. Rolling Volatility of Five-Minute XAU/USD Returns**


## 9. Figure III-4 — Distribution of Five-Minute XAU/USD Returns

The return distribution provides an exploratory view of the central tendency, dispersion, and extreme observations in the five-minute return series.


In [ ]:
returns = df["log_return"].dropna()

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(returns, bins=150)
ax.axvline(
    returns.mean(),
    linewidth=1.0,
    linestyle="--",
)
ax.set_title("Distribution of Five-Minute XAU/USD Returns")
ax.set_xlabel("Log Return")
ax.set_ylabel("Frequency")
ax.grid(True, alpha=0.2)
fig.tight_layout()
plt.show()


**Figure III-4. Distribution of Five-Minute XAU/USD Returns**


## 10. Exploratory return statistics

The following descriptive statistics summarize the five-minute return series used in the preliminary visualization.


In [ ]:
return_stats = returns.describe()

summary = return_stats.to_frame(name="Value")
summary.loc["skewness"] = returns.skew()
summary.loc["kurtosis"] = returns.kurt()

display(summary)


## 11. Interpretation notes

- Figure III-1 provides the broad historical trajectory of XAU/USD closing prices.
- Figure III-2 emphasizes short-term price changes and extreme return observations.
- Figure III-3 shows changing local volatility over the historical period.
- Figure III-4 shows the empirical distribution of five-minute returns and provides a preliminary view of dispersion and extreme observations.

These figures support initial dataset characterization only. They do not determine the final anomaly threshold, do not represent LGMMA-X anomaly scores, and do not constitute the final model evaluation.
